# ML Methods Comparison

This notebook creates radar plots comparing different ML classifiers within one model family.

Section 1 is for the clinical model. You only need to fill the configuration block, then run all following cells.

## Section 1. Clinical Model Radar Plot

Goal: compare RF, LR, XGBoost, and SVM AUCs in internal test and external test.

- If a method is the final-selection method, the notebook reads from `clinical/final_selections`.
- For the remaining methods, manually define one experiment folder, such as `random30_rfecv_none`.
- For each cohort, choose which probability to use: `final`, `mean`, `alldata`, or `best`.


In [ ]:
# ============================================================
# 1. Imports and global paths
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

MODEL_ROOT = Path('/host/d/projects/Habitats/models/Prognosis')
RESULTS_ROOT = Path('/host/d/projects/Habitats/results')
OUT_DIR = RESULTS_ROOT / 'ML_methods_comparison'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLINICAL_ROOT = MODEL_ROOT / 'clinical'
CLINICAL_FINAL_DIR = CLINICAL_ROOT / 'final_selections'

LABEL_COL = 'Prognosis_label'
ID_COLS = ['Patient_set', 'Patient_index']

print('Clinical root:', CLINICAL_ROOT)
print('Output dir:', OUT_DIR)


In [ ]:
# ============================================================
# 2. User settings: fill this block before running
# ============================================================

# Order around the radar plot.
METHOD_ORDER = ['RF', 'LR', 'XGBoost', 'SVM']

# Folder names under /models/Prognosis/clinical.
# If your XGBoost folder has a different name later, change it here.
METHOD_DIR_MAP = {
    'RF': 'RandomForest',
    'LR': 'LR',
    'XGBoost': 'XGBoost',
    'SVM': 'SVM',
}

# Which ML method is the final-selection method for clinical model?
# Example from current clinical final_selection is likely 'LR'.
FINAL_SELECTION_METHOD = 'LR'

# For methods that are NOT final selection, manually define the single experiment folder to use.
# Leave the final-selection method value blank or None; it will read from final_selections.
# Example experiment names: random30_rfecv_none, random20_lasso_none, random10_none_none
MANUAL_EXPERIMENTS = {
    'RF': '',       # e.g. 'random30_rfecv_none'
    'LR': '',       # final selection method, usually leave blank
    'XGBoost': '',  # e.g. 'random30_lasso_none'
    'SVM': '',      # e.g. 'random20_rfecv_none'
}

# Which probability column should be used for each method and cohort?
# Allowed values:
#   final   -> prob_final for ordinary experiment; prob_final_selection for final_selection files
#   mean    -> prob_mean
#   alldata -> prob_alldata, ordinary experiment only
#   best    -> prob_best, ordinary experiment only
#
# For final-selection files, 'final' is usually the clean choice.
INTERNAL_PROB_MODE = {
    'RF': 'final',
    'LR': 'final',
    'XGBoost': 'final',
    'SVM': 'final',
}

EXTERNAL_PROB_MODE = {
    'RF': 'final',
    'LR': 'final',
    'XGBoost': 'final',
    'SVM': 'final',
}

# Plot settings.
FIGSIZE = (6.8, 6.2)
Y_MIN = 0.0
Y_MAX = 1.0
Y_TICKS = [0.2, 0.4, 0.6, 0.8, 1.0]
COLORS = {
    'Internal test': '#1f77b4',
    'External test': '#d62728',
}

SAVE_PREFIX = 'clinical_ML_methods_AUC_radar'


In [ ]:
# ============================================================
# 3. Helper functions
# ============================================================

def normalize_method_name(method):
    if method not in METHOD_ORDER:
        raise ValueError(f'Unknown method: {method}. Expected one of {METHOD_ORDER}')
    return method


def prediction_file_for_experiment(method, experiment, cohort):
    method = normalize_method_name(method)
    if not experiment:
        raise ValueError(
            f'MANUAL_EXPERIMENTS["{method}"] is empty. '
            f'Fill it unless {method} is FINAL_SELECTION_METHOD.'
        )
    method_dir = METHOD_DIR_MAP[method]
    filename = 'internal_test_predictions.xlsx' if cohort == 'internal' else 'external_test_predictions.xlsx'
    path = CLINICAL_ROOT / method_dir / experiment / filename
    if not path.exists():
        raise FileNotFoundError(f'Prediction file not found: {path}')
    return path


def final_selection_prediction_file(cohort):
    filename = 'internal_test_final_selection_predictions.xlsx' if cohort == 'internal' else 'external_test_final_selection_predictions.xlsx'
    path = CLINICAL_FINAL_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Final-selection prediction file not found: {path}')
    return path


def probability_column_for_mode(df, mode, is_final_selection=False):
    mode = str(mode).lower().strip()
    if is_final_selection:
        if mode in {'final', 'final_selection', 'selection'}:
            candidates = ['prob_final_selection']
        elif mode == 'mean':
            candidates = ['prob_mean']
        else:
            raise ValueError(
                f'For final-selection files, mode={mode!r} is not directly available. '
                'Use final or mean, or read the underlying ordinary experiment instead.'
            )
    else:
        mapping = {
            'final': ['prob_final'],
            'mean': ['prob_mean'],
            'alldata': ['prob_alldata'],
            'best': ['prob_best'],
        }
        if mode not in mapping:
            raise ValueError(f'Unknown probability mode: {mode}. Allowed: final, mean, alldata, best')
        candidates = mapping[mode]

    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f'No probability column found for mode={mode}. Tried {candidates}. Available columns: {df.columns.tolist()}')


def read_prediction(method, cohort, mode):
    method = normalize_method_name(method)
    is_final = (method == FINAL_SELECTION_METHOD)

    if is_final:
        path = final_selection_prediction_file(cohort)
        source = f'final_selection/{path.name}'
    else:
        experiment = MANUAL_EXPERIMENTS[method]
        path = prediction_file_for_experiment(method, experiment, cohort)
        source = f'{METHOD_DIR_MAP[method]}/{experiment}/{path.name}'

    df = pd.read_excel(path)
    prob_col = probability_column_for_mode(df, mode, is_final_selection=is_final)

    if LABEL_COL not in df.columns:
        raise KeyError(f'{LABEL_COL} not found in {path}')

    y = df[LABEL_COL].astype(int).to_numpy()
    p = df[prob_col].astype(float).to_numpy()
    auc = roc_auc_score(y, p)

    info = {
        'Method': method,
        'Cohort': 'Internal test' if cohort == 'internal' else 'External test',
        'Is_final_selection_method': is_final,
        'Experiment': 'final_selection' if is_final else MANUAL_EXPERIMENTS[method],
        'Probability_mode': mode,
        'Probability_column': prob_col,
        'Prediction_file': str(path),
        'AUC': auc,
        'N': len(df),
        'Positive_n': int(y.sum()),
        'Positive_fraction': float(y.mean()),
        'Source': source,
    }
    return df, info


def collect_auc_table():
    rows = []
    for method in METHOD_ORDER:
        _, internal_info = read_prediction(method, 'internal', INTERNAL_PROB_MODE[method])
        _, external_info = read_prediction(method, 'external', EXTERNAL_PROB_MODE[method])
        rows.extend([internal_info, external_info])
    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 4. Read predictions and calculate AUCs
# ============================================================

auc_long_df = collect_auc_table()
auc_wide_df = auc_long_df.pivot(index='Method', columns='Cohort', values='AUC').reindex(METHOD_ORDER)

save_long_path = OUT_DIR / f'{SAVE_PREFIX}_auc_long.xlsx'
save_wide_path = OUT_DIR / f'{SAVE_PREFIX}_auc_wide.xlsx'
auc_long_df.to_excel(save_long_path, index=False)
auc_wide_df.reset_index().to_excel(save_wide_path, index=False)

print('Saved:', save_long_path)
print('Saved:', save_wide_path)
display(auc_long_df)
display(auc_wide_df)


In [ ]:
# ============================================================
# 5. Radar plot
# ============================================================

def close_loop(values):
    values = list(values)
    return values + values[:1]

labels = METHOD_ORDER
n = len(labels)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles_closed = angles + angles[:1]

fig = plt.figure(figsize=FIGSIZE)
ax = plt.subplot(111, polar=True)

for cohort in ['Internal test', 'External test']:
    values = auc_wide_df[cohort].astype(float).to_list()
    ax.plot(
        angles_closed,
        close_loop(values),
        color=COLORS[cohort],
        linewidth=2.2,
        label=cohort,
    )
    ax.fill(
        angles_closed,
        close_loop(values),
        color=COLORS[cohort],
        alpha=0.18,
    )

ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=13)
ax.set_ylim(Y_MIN, Y_MAX)
ax.set_yticks(Y_TICKS)
ax.set_yticklabels([f'{v:.1f}' for v in Y_TICKS], fontsize=11)
ax.set_rlabel_position(90)
ax.grid(True, linestyle=':', linewidth=0.8, alpha=0.8)
ax.spines['polar'].set_color('#888888')
ax.spines['polar'].set_linewidth(0.9)

ax.set_title('Clinical model', y=1.08, fontsize=16, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.28, 1.10), frameon=False, fontsize=12)

# AUC labels near each marker.
for cohort in ['Internal test', 'External test']:
    values = auc_wide_df[cohort].astype(float).to_list()
    for angle, value in zip(angles, values):
        ax.text(
            angle,
            min(value + 0.045, Y_MAX),
            f'{value:.3f}',
            color=COLORS[cohort],
            fontsize=10,
            ha='center',
            va='center',
        )

plt.tight_layout()

pdf_path = OUT_DIR / f'{SAVE_PREFIX}.pdf'
png_path = OUT_DIR / f'{SAVE_PREFIX}.png'
fig.savefig(pdf_path, bbox_inches='tight')
fig.savefig(png_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved radar plot:', pdf_path)
print('Saved radar preview:', png_path)
